In [36]:
# import libraries
import re
import heapq
import string
from tqdm import tqdm
import numpy as np
import pandas as pd
from urllib.request import urlopen

# if you need to plot anything
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt

# Stopword dictionary
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk import download, wordnet
nltk.download('omw-1.4')
download('punkt')
nltk.download('punkt_tab')
download('stopwords')
download("wordnet")



# For stemming
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv = CountVectorizer() # the Bag of Words
tfidf = TfidfVectorizer()

# let's import ML algorithms for classification
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split as tts, KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score as ac, confusion_matrix as cm, ConfusionMatrixDisplay as CMD

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/dvasiliu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dvasiliu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [37]:
# the first example uses only n observations
wine_data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/winemagdata130kv2.csv?raw=true',quoting=2)
wines = wine_data[["description","points"]]
# wines_subset = wines.sample(5000,random_state=1234).reset_index(drop=True)
corpus = []

In [4]:
wines

,description,points
0,"Aromas include tropical fruit, broom, brimston...",87.0
1,"This is ripe and fruity, a wine that is smooth...",87.0
2,"Tart and snappy, the flavors of lime flesh and...",87.0
3,"Pineapple rind, lemon pith and orange blossom ...",87.0
4,"Much like the regular bottling from 2012, this...",87.0
...,...,...
129966,Notes of honeysuckle and cantaloupe sweeten th...,90.0
129967,Citation is given as much as a decade of bottl...,90.0
129968,Well-drained gravel soil gives this wine its c...,90.0
129969,"A dry style of Pinot Gris, this is crisp with ...",90.0


In [6]:
reviews = wines['description'].values

In [7]:
reviews[0]

"Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity."

In [9]:
reviews[129966]

"Notes of honeysuckle and cantaloupe sweeten this deliciously feather-light spätlese. It's intensely juicy, quenching the palate with streams of tart tangerine and grapefruit acidity, yet wraps up with a kiss of honey and peach."

In [11]:
# you can do this but you will be frustated with the runtime!!!
prep_reviews = []
for review in tqdm(reviews):
    wine_descriptions = re.sub('[^a-zA-Z0-9 ]','',review)
    wine_descriptions=wine_descriptions.lower()
    wine_descriptions = wine_descriptions.split()
    wine_descriptions = [word for word in wine_descriptions if not word in set(stopwords.words('english'))]
    stemmer = PorterStemmer()
    wine_descriptions = [stemmer.stem(word) for word in wine_descriptions]
    wine_descriptions = " ".join(wine_descriptions)
    prep_reviews.append(wine_descriptions)

100%|██████████| 129971/129971 [02:30<00:00, 866.17it/s]


In [38]:
def fast_preprocess(texts, lemmatize=True):

    # Fast text preprocessing function that handles:
    # - Lowercasing
    # - Punctuation removal
    # - Stopword removal
    # - Lemmatization (optional)

    # Arguments:
    #     texts: List of strings or single string to process
    #     lemmatize: Whether to perform lemmatization

    # Returns:
    #     Processed texts (list or string, matching input type)

    # Handle single string input
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]



    stop_words = set(stopwords.words('english'))

    # Create translator for punctuation removal (faster than regex for this task)
    translator = str.maketrans('', '', string.punctuation)

    # Initialize lemmatizer if needed
    lemmatizer = WordNetLemmatizer() if lemmatize else None

    processed_texts = []
    for text in texts:
        # Lowercase
        text = text.lower()

        # Remove punctuation
        text = text.translate(translator)

        # Tokenize (needed for both stopword removal and lemmatization)
        tokens = text.split()

        # Remove stopwords
        tokens = [token for token in tokens if token not in stop_words]

        # Lemmatize if requested
        if lemmatize:
            tokens = [lemmatizer.lemmatize(token) for token in tokens]

        # Rejoin into text
        processed_texts.append(' '.join(tokens))

    # Return in same format as input
    return processed_texts[0] if single_input else processed_texts

In [39]:
prep_reviews_lem = fast_preprocess(reviews)

In [14]:
prep_reviews[129966]

'note honeysuckl cantaloup sweeten delici featherlight sptlese intens juici quench palat stream tart tangerin grapefruit acid yet wrap kiss honey peach'

In [16]:
prep_reviews_lem[129966]

'note honeysuckle cantaloupe sweeten deliciously featherlight spätlese intensely juicy quenching palate stream tart tangerine grapefruit acidity yet wrap kiss honey peach'

In [18]:
# if we pretend the whole data is the train data
cv = CountVectorizer()
x_bow = cv.fit_transform(prep_reviews_lem)

In [40]:
y = wines['points'].values

In [21]:
y

array([87., 87., 87., ..., 90., 90., 90.], shape=(129971,))

In [22]:
x_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3160873 stored elements and shape (129971, 39929)>

In [23]:
# we can think about the snetiment analysis in different ways, regression or classification
from sklearn.linear_model import ElasticNet
model = ElasticNet(alpha=0.001,l1_ratio=0.5,max_iter=2000)
model.fit(x_bow,y)

ElasticNet(alpha=0.001, max_iter=2000)

In [24]:
model.score(x_bow,y)

0.6643718211384677

In [27]:
print(f'The RMSE value by using the whole data is: {np.sqrt(np.mean((y-model.predict(x_bow))**2))}')

The RMSE value by using the whole data is: 1.761013057423535


In [28]:
print(f'The MAE value by using the whole data is: {np.mean(np.abs((y-model.predict(x_bow))))}')

The MAE value by using the whole data is: 1.390622358350339


In [29]:
# I want an unbiased estimation of the MSE
from sklearn.pipeline import Pipeline

In [31]:
pipe = Pipeline([['BoW',CountVectorizer()],['Regressor',ElasticNet(alpha=0.001,l1_ratio=0.5)]])

In [ ]:
cv = KFold(n_splits=5,shuffle=True,random_state=301)
abs(cross_val_score(pipe,prep_reviews_lem,y,cv=cv,scoring='neg_mean_squared_error'))

In [ ]:
## We can consider our sentiment analysis as a classification problem
yb = (y>=90) + 0

In [42]:
pts = wines.points
yb = pts.copy().values
yb[pts>=92] = 2
yb[(pts>=89) & (pts<=91)] = 1
yb[(pts<89)] = 0


In [44]:
sum(yb==2)/len(yb)

np.float64(0.17139207977164136)

In [45]:
# we'll use here only a train/test split for speed
xtrain, xtest, ytrain, ytest = tts(prep_reviews_lem,yb,test_size=0.25,shuffle=True,random_state=301)

In [47]:
# we use the data ethically
xtrain_v = tfidf.fit_transform(xtrain)
xtest_v = tfidf.transform(xtest)

In [52]:
model = LogisticRegression(C=5,solver='saga',max_iter=2000)
model.fit(xtrain_v,ytrain)
model.score(xtest_v,ytest)

0.7290493337026437

In [53]:
from sklearn.metrics import classification_report as CR

In [57]:
print(CR(ytest,model.predict(xtest_v)))

              precision    recall  f1-score   support

         0.0       0.82      0.86      0.84     17235
         1.0       0.58      0.57      0.57      9679
         2.0       0.69      0.60      0.64      5579

    accuracy                           0.73     32493
   macro avg       0.69      0.68      0.68     32493
weighted avg       0.72      0.73      0.73     32493



In [60]:
model = RandomForestClassifier(n_estimators=200,max_depth=50)
model.fit(xtrain_v,ytrain)
model.score(xtest_v,ytest)

0.6406918413196688